# PneumoniaMNIST + MedSymmFlow Synthetic Augmentation

**Does augmenting PneumoniaMNIST with MedSymmFlow-generated images improve a ResNet-18 classifier on the held-out test split?**

Runnable core of the v2.0 augmentation protocol. The experiment logic lives in
`project/augmentation.py` **in the repo**, so fixes arrive via `git pull` — this
notebook is just config, narrative, and calls.

| Protocol item | Implemented |
|---|---|
| **G1 split hygiene** — explicit `val`(524)/`test`(624), never select on test | genuine `medmnist` splits, asserted |
| **G3** — 28px, **MSF** generator | Zenodo `RGB_28` checkpoint (sampled at 32, stored at 28) |
| **Arms** | baselines **B0/B1/B2**, synthetic **S1/S2/S3**, **C1** reference |
| **Data-scaling** | `budgets` sweep, stratified subsampling, fixed seeds |
| **Filtering (Sec 7)** | memorisation NN screen + confidence filter |
| **Evaluation** | test **AUC** primary, val-fixed threshold, multi-seed 95% CIs |
| **Speed** | mixed precision (`use_amp`) |

> **Run order:** top to bottom. `Config(quick=True)` is a fast smoke test; set `quick=False` for the real run (3 seeds -> real CIs). After a `git pull` that changes the module, do **Runtime > Restart session** and re-run so the reimport picks up the edits.

## 0. Setup — mount, clone, install

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone/update the fork that holds the fixed generator + the experiment module.
%cd /content
![ -d MedSymmFlow ] || git clone https://github.com/RonNekrashevich/MedSymmFlow.git
%cd /content/MedSymmFlow
!git pull -q

In [ ]:
!pip install -q medmnist torchdiffeq diffusers accelerate zuko scikit-learn loguru python-dotenv datasets

In [ ]:
# Preflight: import the MedSymmFlow chain exactly as the generator subprocess does.
# Fails in seconds if a dependency is missing, before any training runs.
import subprocess, os
_probe = ("from medsymmflow.models.SymmFMClass import SymmFMClass; "
          "from medsymmflow.data.Dataloaders import pick_dataset; print('MedSymmFlow imports OK')")
_res = subprocess.run(["python", "-c", _probe], cwd="/content/MedSymmFlow",
                      env=dict(os.environ, PYTHONPATH="/content/MedSymmFlow/src:/content/MedSymmFlow/src/medsymmflow"),
                      capture_output=True, text=True)
print(_res.stdout.strip() or _res.stderr[-1500:])
assert _res.returncode == 0, "MedSymmFlow import chain is broken - fix before continuing"

In [ ]:
# Import the experiment module from the repo. After a `git pull` that changes
# augmentation.py, do Runtime > Restart session and re-run to pick up the edits.
import sys
sys.path.insert(0, "/content/MedSymmFlow/project")
import augmentation
from augmentation import Experiment, Config

exp = Experiment(Config(quick=True))   # quick=False for the real run

## 1. Data & split hygiene (G1)

Loads the **genuine** `medmnist` splits (sidestepping the repo loaders that alias val->test), asserts 4708/524/624, and reports the train/test prevalence shift — the reason AUC (threshold-free) is primary.

In [ ]:
prevalence = exp.setup_data()
display(prevalence)

## 2. Baseline arms — B0 / B1 / B2

All real-data baselines, so a synthetic arm must beat the *strongest* of them:
**B0** plain, **B1** class-weighted loss, **B2** minority oversampling. B0 (full budget, seed 0) is reused as the confidence-filter scorer.

In [ ]:
exp.run_baselines()

## 3. Synthetic generation — MSF @ 28 px

Downloads the pretrained MSF weights (Zenodo) and samples class-conditioned images. The RGB_28 checkpoint is sampled at 32px (its training resolution / UNet divisibility) and stored at 28px to match the real data. Everything persists to Drive.

In [ ]:
exp.download_weights()
exp.generate_synthetic()
exp.visualize_samples()   # Sec 7.3 visual check: should look like chest X-rays

## 4. Filtering (Sec 7)

**Memorisation screen** (discard synthetic near-copies of real training images via a fixed ImageNet encoder) + **confidence filter** (keep only samples the real-only B0 model scores confidently and consistently).

In [ ]:
exp.filter_synthetic()

## 5. Synthetic arms — S1 / S2 / S3

**S1** pretrain->fine-tune, **S2** naive real+synthetic mixing, **S3** minority-only synthetic to rebalance prevalence. All keep standard augmentation and select on `val`.

In [ ]:
exp.run_synthetic()

## 5b. Diagnostic D1 — synthetic-only (distillation probe)

Train on synthetic images **only**, test on real. If D1 recovers baseline/C1-level AUC, the synthetic set alone carries the decision function — evidence that the S1/S2 gains are **distillation** of MSF's boundary, not new information. If D1 collapses, the gains are data-manifold coverage instead.

In [ ]:
exp.run_diagnostic_d1()

## 6. C1 — MedSymmFlow reference (distillation control)

MSF scores AUC 0.952 at 28px — above ResNet-18's 0.944. If any synthetic arm beats the baselines, distillation of MSF's decision function is the leading explanation, so C1 is reported alongside every arm.

In [ ]:
exp.record_c1()

## 7. Results & comparison

Per (arm, budget): mean test AUC with 95% CI across seeds (needs >=2 seeds — `quick=False`). Decision rule: a synthetic arm is effective only if it beats the strongest baseline with **non-overlapping CIs** *and* the gain isn't fully explained by C1.

In [ ]:
summary, comparison = exp.summarize()
display(summary)
print("\nSynthetic arms vs strongest baseline (C1 =", exp.cfg.c1_auc, "):")
display(comparison)
exp.plot(summary)

## 7b. Distillation fingerprint — does synthetic training copy MSF's errors?

The decisive probe. We run **MSF's own** reverse-flow classifier on the real test split, then check whether the **synthetic-trained** ResNet (D1) agrees with MSF's predictions — *especially where MSF is wrong* — more than the **real-trained** ResNet (B0). Copying a model's specific errors requires copying its decision function, which data-manifold coverage alone cannot produce. Higher `agree_on_MSF_errors` for the synthetic-trained model is a distillation signature.

*(The MSF-classification path runs the same model as generation, via subprocess; it's the one part not GPU-tested during authoring — if it errors, the rest of the results above are unaffected.)*

In [ ]:
fingerprint = exp.distillation_agreement()
display(fingerprint)

## 8. Extensions to the full protocol

- **Full sweep:** `Config(quick=False, budgets=[250,500,1000,2000,4708], seeds=[0,1,2,3,4])`. **Compute:** 6 arms x 5 budgets x 5 seeds is many hours — trim, or checkpoint `results.csv` per budget.
- **Generation sweeps:** `Config(...).gen_beta` in {1,2,4,6}, ODE steps in {10,25,50}, synthetic:real ratio — select on `val`.
- **Statistics:** CIs are computed; add a paired seed test + Benjamini-Hochberg correction across the sweep.
- **224px / LatMSF:** reserve for configs that survive at 28px.

Now implemented: **D1** (synthetic-only diagnostic) and the **MSF distillation fingerprint** (C1 is reproduced live by `classify_pneumoniamnist.py`, not just the published constant).

**Reporting reminders:** B0 must reproduce ~0.944 AUC before interpreting any synthetic arm; report C1 next to every synthetic result; never select on `test`.

**Editing the logic:** change `project/augmentation.py` in the repo, run the `git pull` cell, then **Runtime > Restart session** and re-run. No notebook re-upload needed for logic fixes.